In [8]:
pip install pandas numpy scikit-learn tensorflow transformers tf-keras

Note: you may need to restart the kernel to use updated packages.


In [3]:
# ------------------------------------------------------------
# 0) Install (ONLY if needed - run once)
# ------------------------------------------------------------
# !pip install -U transformers datasets

# ------------------------------------------------------------
# 1) Imports
# ------------------------------------------------------------
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import pandas as pd
import numpy as np
import torch

# ------------------------------------------------------------
# 2) Load Data
# ------------------------------------------------------------
df = pd.read_csv("/kaggle/input/datasets/shuvo128/electra/IMDB_Cleaned (1).csv")

text_column = "cleaned_review" if "cleaned_review" in df.columns else "review"
texts  = df[text_column].astype(str).tolist()
labels = (df["sentiment"] == "positive").astype(int).tolist()

# ------------------------------------------------------------
# 3) Split (70 / 20 / 10)
# ------------------------------------------------------------
train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    texts, labels, test_size=0.3, random_state=42, stratify=labels
)

val_texts, test_texts, val_labels, test_labels = train_test_split(
    temp_texts, temp_labels, test_size=1/3, random_state=42, stratify=temp_labels
)

print(f"Train: {len(train_texts)}, Val: {len(val_texts)}, Test: {len(test_texts)}")

# ------------------------------------------------------------
# 4) Tokenizer
# ------------------------------------------------------------
tokenizer = AutoTokenizer.from_pretrained("google/electra-base-discriminator")

def tokenize(texts):
    return tokenizer(texts, truncation=True, padding=True, max_length=256)

train_enc = tokenize(train_texts)
val_enc   = tokenize(val_texts)
test_enc  = tokenize(test_texts)

# ------------------------------------------------------------
# 5) Dataset Class
# ------------------------------------------------------------
class Dataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = Dataset(train_enc, train_labels)
val_dataset   = Dataset(val_enc, val_labels)
test_dataset  = Dataset(test_enc, test_labels)

# ------------------------------------------------------------
# 6) Model (ELECTRA)
# ------------------------------------------------------------
model = AutoModelForSequenceClassification.from_pretrained(
    "google/electra-base-discriminator",
    num_labels=2
)

# ------------------------------------------------------------
# 7) Metrics (VERY IMPORTANT for paper 🔥)
# ------------------------------------------------------------
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average='binary'
    )
    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

# ------------------------------------------------------------
# 8) Training Arguments (compatible version)
# ------------------------------------------------------------
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    logging_steps=100,
    save_strategy="no",
    report_to="none"
)

# ------------------------------------------------------------
# 9) Trainer
# ------------------------------------------------------------
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

# ------------------------------------------------------------
# 10) Train
# ------------------------------------------------------------
trainer.train()

# ------------------------------------------------------------
# 11) Validation Results
# ------------------------------------------------------------
print("\nValidation Results:")
val_results = trainer.evaluate()
print(val_results)

# ------------------------------------------------------------
# 12) Test Results (IMPORTANT)
# ------------------------------------------------------------
print("\nTest Results:")
test_results = trainer.predict(test_dataset)

test_preds = np.argmax(test_results.predictions, axis=1)

precision, recall, f1, _ = precision_recall_fscore_support(
    test_labels, test_preds, average='binary'
)
acc = accuracy_score(test_labels, test_preds)

print({
    "accuracy": acc,
    "precision": precision,
    "recall": recall,
    "f1": f1
})

Train: 35000, Val: 10000, Test: 5000


config.json:   0%|          | 0.00/666 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Step,Training Loss
100,0.763273
200,0.483247
300,0.418088
400,0.417882
500,0.449332
600,0.356937
700,0.380885
800,0.377907
900,0.360998
1000,0.374773



Validation Results:


{'eval_loss': 0.5043848752975464, 'eval_accuracy': 0.9408, 'eval_precision': 0.9373015873015873, 'eval_recall': 0.9448, 'eval_f1': 0.9410358565737051, 'eval_runtime': 94.6885, 'eval_samples_per_second': 105.609, 'eval_steps_per_second': 3.306, 'epoch': 3.0}

Test Results:


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{'accuracy': 0.9448, 'precision': 0.9465863453815261, 'recall': 0.9428, 'f1': 0.944689378757515}
